<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [149]:
#This is code to manage dependencies if the notebook is executed in the google colab cloud service
if 'google.colab' in str(get_ipython()):
  import os
  os.system('apt -qqq install graphviz')
  os.system('pip -qqq install ModelFlowIb')

In [150]:
import pandas as pd
import re

from modelclass import model
import modelmanipulation as mp
from modelmanipulation import doable, explode

from modelpattern import list_extract
import modeljupytermagic

In [151]:
from modelhelp import colab_link
colab_link('Specification_debt_simulation_model',badge=True,render=0)

<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# About lists

**lists** and **sublists** are central to making ModelFlow scalable and expressive.  
They allow the modeler to define **groups of entities** (such as bonds, banks, portfolios, sectors, or countries) and to attach **attributes** or **hierarchies** to those entities.  
A list definition typically takes the following form

$list\: \underbrace{issued}_{List name} = \underbrace{issued}_{sub listname} : issued\_2025 * issued\_2050 / \\
\underbrace{issued\_year}_{sub listname} : 2025 * 2050$

Please note:
 - the list name and the first sub list name should be the same
 - case don't matter
 - a * combined with identifiers ending in digits will construct multiple list content

  `issued_2025 * issued_2050` becomes `issued_2025 issued_2026 … issued_2050`


This enables equations to be written once as templates and then **expanded automatically** across all relevant list members.  

Sublists can be used to define logical relationships between entities — for example, linking each bond to its home maturity, or to define the year for each issurance year.  
These structures make it possible to organize models systematically and to maintain consistency across entities that share the same behavioral equations.  

This approach eliminates redundancy and manual repetition: instead of writing dozens or hundreds of similar equations, the modeler defines a **single equation pattern**, and ModelFlow expands it dynamically over the members of a list.  

In [152]:
%%Mexplodemodel hist segment=listbonds
# List definitions


> list issued = issued        : issued_2025  /
>>              issued_year   :        2025 

>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000  /
>>              10_hist_usd        10      0       usd         0        1    2          200

>list currency = currency : dom usd    

# List definitions


```text
> list issued = issued        : issued_2025  /
>>              issued_year   :        2025 
```

```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000  /
>>              10_hist_usd        10      0       usd         0        1    2          200
```

```text
>list currency = currency : dom usd    
```

In [153]:
port_replacements=[('__bond','__{bond}__{issued}'),
                   ('fx_rate','dom_{currency}')
                  ]

In [154]:
%%Mexplodemodel hist segment=logical replacements=port_replacements nshow render=0 render_list=False

## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

> doable logical_issuing__bond   = current_year == {issued_year}
> doable logical_maturing__bond  =  current_year == {issued_year}+{maturity}
> doable logical_amortizing_years__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})
> doable logical_paying_years__bond     =  {issued_year}            < current_year <= ({issued_year} + {maturity})


In [155]:
%%Mexplodemodel hist segment=stock replacements=port_replacements  show render=0 render_list=False


> doable amortizing_share_pr_time__bond =  logical_amortizing_years__bond / ({maturity}-{grace})
### Amortizing in fx 
> doable  amortizing_in_currency__bond = {start_in_currency} * amortizing_share_pr_time__bond

### End of year stock in fx
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond-amortizing_in_currency__bond

### Start of year stock in fx 
> doable stock_primo_in_currency__bond = (current_year=={issued_year}+1)*{start_in_currency} +stock_ultimo_in_currency__bond(-1)




FRML <> AMORTIZING_SHARE_PR_TIME__10_HIST_DOM__ISSUED_2025 = LOGICAL_AMORTIZING_YEARS__10_HIST_DOM__ISSUED_2025/(10-0) $
FRML <> AMORTIZING_SHARE_PR_TIME__10_HIST_USD__ISSUED_2025 = LOGICAL_AMORTIZING_YEARS__10_HIST_USD__ISSUED_2025/(10-0) $
FRML <> AMORTIZING_IN_CURRENCY__10_HIST_DOM__ISSUED_2025 = 1000*AMORTIZING_SHARE_PR_TIME__10_HIST_DOM__ISSUED_2025 $
FRML <> AMORTIZING_IN_CURRENCY__10_HIST_USD__ISSUED_2025 = 200*AMORTIZING_SHARE_PR_TIME__10_HIST_USD__ISSUED_2025 $
FRML <> STOCK_ULTIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025 = STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025-AMORTIZING_IN_CURRENCY__10_HIST_DOM__ISSUED_2025 $
FRML <> STOCK_ULTIMO_IN_CURRENCY__10_HIST_USD__ISSUED_2025 = STOCK_PRIMO_IN_CURRENCY__10_HIST_USD__ISSUED_2025-AMORTIZING_IN_CURRENCY__10_HIST_USD__ISSUED_2025 $
FRML <> STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025 = (CURRENT_YEAR==2025+1)*1000+STOCK_ULTIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025(-1) $
FRML <> STOCK_PRIMO_IN_CURRENCY__10_HIST_USD__ISSUED_202

In [156]:
%%Mexplodemodel hist segment=interest_rate replacements=port_replacements  nshow render=0 render_list=False

### Interest rate payment
> doable interest_in_currency__bond = stock_primo_in_currency__bond * {rate}/100 *  logical_paying_years__bond

In [157]:
%%Mexplodemodel hist segment=sums replacements=port_replacements  nshow render=0 render_list=False
## portfoliosums
### Amortizing in domestic currency 

> doable <sum=all>  amortizing__bond = amortizing_in_currency__bond * fx_rate

### Stock in domestic currency 
> doable <sum=all> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=all> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate

### Interest rate payment

> doable <sum=all>  interest_payments__bond    = interest_in_currency__bond  * fx_rate

In [158]:
%Mexplodemodel hist replacements=port_replacements
mhist = hist.mmodel


## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

```text
> doable logical_issuing__bond   = current_year == {issued_year}
> doable logical_maturing__bond  =  current_year == {issued_year}+{maturity}
> doable logical_amortizing_years__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})
> doable logical_paying_years__bond     =  {issued_year}            < current_year <= ({issued_year} + {maturity})
```



```text
> doable amortizing_share_pr_time__bond =  logical_amortizing_years__bond / ({maturity}-{grace})
```
### Amortizing in fx 
```text
> doable  amortizing_in_currency__bond = {start_in_currency} * amortizing_share_pr_time__bond
```

### End of year stock in fx
```text
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond-amortizing_in_currency__bond
```

### Start of year stock in fx 
```text
> doable stock_primo_in_currency__bond = (current_year=={issued_year}+1)*{start_in_currency} +stock_ultimo_in_currency__bond(-1)
```




### Interest rate payment
```text
> doable interest_in_currency__bond = stock_primo_in_currency__bond * {rate}/100 *  logical_paying_years__bond
```

## portfoliosums
### Amortizing in domestic currency 

```text
> doable <sum=all>  amortizing__bond = amortizing_in_currency__bond * fx_rate
```

### Stock in domestic currency 
```text
> doable <sum=all> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=all> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate
```

### Interest rate payment

```text
> doable <sum=all>  interest_payments__bond    = interest_in_currency__bond  * fx_rate
```

# List definitions


```text
> list issued = issued        : issued_2025  /
>>              issued_year   :        2025 
```

```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000  /
>>              10_hist_usd        10      0       usd         0        1    2          200
```

```text
>list currency = currency : dom usd    
```

In [159]:
years = [y for y in range(2025,2050,1)]
dfstart = pd.DataFrame(years,index=years,columns=['CURRENT_YEAR'])
dfstart

,CURRENT_YEAR
2025,2025
2026,2026
2027,2027
2028,2028
2029,2029
2030,2030
2031,2031
2032,2032
2033,2033
2034,2034


In [168]:
mhist.exogene

{'CURRENT_YEAR', 'DOM_DOM', 'DOM_USD'}

In [176]:
df = dfstart.upd('''
dom_dom = 1
dom_usd = 10
''')
df.T


,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
CURRENT_YEAR,2025.0,2026.0,2027.0,2028.0,2029.0,2030.0,2031.0,2032.0,2033.0,2034.0,...,2040.0,2041.0,2042.0,2043.0,2044.0,2045.0,2046.0,2047.0,2048.0,2049.0
DOM_DOM,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
DOM_USD,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,...,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0


In [177]:
res = mhist(df,2026,2050,silent=0)

Will start calculating: testmodel
2026  solved
2027  solved
2028  solved
2029  solved
2030  solved
2031  solved
2032  solved
2033  solved
2034  solved
2035  solved
2036  solved
2037  solved
2038  solved
2039  solved
2040  solved
2041  solved
2042  solved
2043  solved
2044  solved
2045  solved
2046  solved
2047  solved
2048  solved
2049  solved
testmodel calculated 


In [178]:
with mhist.set_smpl(2025,2050): 
    display(mhist['stock*all interest*all amort*all'].df)

,STOCK_PRIMO__ALL,STOCK_ULTIMO__ALL,INTEREST_PAYMENTS__ALL,AMORTIZING__ALL
2025,0.0,0.0,0.0,0.0
2026,3000.0,2700.0,70.0,300.0
2027,2700.0,2400.0,63.0,300.0
2028,2400.0,2100.0,56.0,300.0
2029,2100.0,1800.0,49.0,300.0
2030,1800.0,1500.0,42.0,300.0
2031,1500.0,1200.0,35.0,300.0
2032,1200.0,900.0,28.0,300.0
2033,900.0,600.0,21.0,300.0
2034,600.0,300.0,14.0,300.0


In [172]:
mhist.STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025

Endogeneous: STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025: 
Formular: FRML <> STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025 = (CURRENT_YEAR==2025+1)*1000+STOCK_ULTIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025(-1) $

STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025 : 
CURRENT_YEAR                                      : 
STOCK_ULTIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025: 

Values :


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
Base,"1,000.00",900.00,800.00,700.00,600.00,500.00,400.00,300.00,200.00,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Last,"1,000.00",900.00,800.00,700.00,600.00,500.00,400.00,300.00,200.00,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Diff,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Input last run:


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
CURRENT_YEAR,"2,026.00","2,027.00","2,028.00","2,029.00","2,030.00","2,031.00","2,032.00","2,033.00","2,034.00","2,035.00","2,036.00","2,037.00","2,038.00","2,039.00","2,040.00","2,041.00","2,042.00","2,043.00","2,044.00","2,045.00","2,046.00","2,047.00","2,048.00","2,049.00"
STOCK_ULTIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025(-1),0.00,900.00,800.00,700.00,600.00,500.00,400.00,300.00,200.00,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [173]:
mhist['stock_primo__all stock_ultimo__all interest*__*26 funding* paying_years__*26'].df.T


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
STOCK_PRIMO__ALL,1200.0,1080.0,960.0,840.0,720.0,600.0,480.0,360.0,240.0,120.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
STOCK_ULTIMO__ALL,1080.0,960.0,840.0,720.0,600.0,480.0,360.0,240.0,120.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [174]:
mhist['amortizing*'].df

,AMORTIZING_IN_CURRENCY__10_HIST_DOM__ISSUED_2025,AMORTIZING_IN_CURRENCY__10_HIST_USD__ISSUED_2025,AMORTIZING_SHARE_PR_TIME__10_HIST_DOM__ISSUED_2025,AMORTIZING_SHARE_PR_TIME__10_HIST_USD__ISSUED_2025,AMORTIZING__10_HIST_DOM__ISSUED_2025,AMORTIZING__10_HIST_USD__ISSUED_2025,AMORTIZING__ALL
2026,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2027,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2028,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2029,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2030,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2031,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2032,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2033,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2034,100.0,20.0,0.1,0.1,100.0,20.0,120.0
2035,100.0,20.0,0.1,0.1,100.0,20.0,120.0


In [175]:
mport.amortizing_in_currency__bond 

NameError: name 'mport' is not defined

In [ ]:
mport.modeldump('model/port')

In [ ]:
mport['amort*__5*'].df.T

In [189]:
mhist.drawmodel()

In [192]:
mhist['#endo']

['AMORTIZING_IN_CURRENCY__10_HIST_DOM__ISSUED_2025',
 'AMORTIZING_IN_CURRENCY__10_HIST_USD__ISSUED_2025',
 'AMORTIZING_SHARE_PR_TIME__10_HIST_DOM__ISSUED_2025',
 'AMORTIZING_SHARE_PR_TIME__10_HIST_USD__ISSUED_2025',
 'AMORTIZING__10_HIST_DOM__ISSUED_2025',
 'AMORTIZING__10_HIST_USD__ISSUED_2025',
 'AMORTIZING__ALL',
 'INTEREST_IN_CURRENCY__10_HIST_DOM__ISSUED_2025',
 'INTEREST_IN_CURRENCY__10_HIST_USD__ISSUED_2025',
 'INTEREST_PAYMENTS__10_HIST_DOM__ISSUED_2025',
 'INTEREST_PAYMENTS__10_HIST_USD__ISSUED_2025',
 'INTEREST_PAYMENTS__ALL',
 'LOGICAL_AMORTIZING_YEARS__10_HIST_DOM__ISSUED_2025',
 'LOGICAL_AMORTIZING_YEARS__10_HIST_USD__ISSUED_2025',
 'LOGICAL_ISSUING__10_HIST_DOM__ISSUED_2025',
 'LOGICAL_ISSUING__10_HIST_USD__ISSUED_2025',
 'LOGICAL_MATURING__10_HIST_DOM__ISSUED_2025',
 'LOGICAL_MATURING__10_HIST_USD__ISSUED_2025',
 'LOGICAL_PAYING_YEARS__10_HIST_DOM__ISSUED_2025',
 'LOGICAL_PAYING_YEARS__10_HIST_USD__ISSUED_2025',
 'STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM__ISSUED_2025',
 'STO